In [1]:
from sodapy import Socrata
import pandas as pd
from itertools import islice
import os
from pathlib import Path

In [3]:
# If the CTA data exists, read it in
if Path("output/cta_ridership.parquet").is_file():
    cta_ridership = pd.read_parquet("output/cta_ridership.parquet")
else:
    # If the CTA data does not exist, download it from Socrata
    client = Socrata(
        "data.cityofchicago.org",
        timeout=1000,
        app_token=None
    )

    cta_data = client.get_all('5neh-572f')

    chunk_size = 10_000
    chunks = []

    while True:
        chunk = list(islice(cta_data, chunk_size))
        print('Grabbing chunk of data...')
        if not chunk:
            break
        chunks.append(pd.DataFrame(chunk))

    cta_df = pd.concat(chunks, ignore_index=True)

    # And save as a flat file
    os.makedirs("output", exist_ok=True)
    cta_df.to_parquet(path='output/cta_ridership.parquet', engine='fastparquet', index=False)

In [ ]:
# Check that the file is up-to-date
# If the data exists and the latest date in the data lags behind the max date on Socrata, download the extra data